Lesson 3: Prompting for structure and setting up a retry method
In this notebook, you'll learn how to combine Pydantic with retry strategies to reliably extract structured output from an LLM.

By the end, you'll be able to:

Define structured data models for LLM responses
Build robust retry mechanisms for validation errors
Create reusable functions for LLM interactions

In [1]:
from google import genai

In [6]:
client = genai.Client()
response = client.models.generate_content(
  model="gemini-2.5-flash",contents="Explain how AI works in few words"
)
print(response.text)

AI learns from data to recognize patterns and make intelligent decisions or predictions.


In [7]:
from pydantic import BaseModel,ValidationError,EmailStr,Field
from typing import Literal,List,Optional
import json
from datetime import date
from dotenv import load_dotenv


In [20]:
#Define a JSON string representing the user input
user_input_json = '''
{
  "name":"Joe Doe",
  "email":"joe.doe@gmail.com",
  "query":"I forgot my password",
  "order_id":null,
  "purchase_date":null
}
'''
type(user_input_json)

str

In [21]:
#Define data model
class UserInput(BaseModel):
  name:str
  email:EmailStr
  query:str
  order_id:Optional[str]=Field(
    None,
    description="5 digit order number",
    ge=10000,
    le=99999
  )
  purchase_date:Optional[date]=None

In [22]:
user_input = UserInput.model_validate_json(user_input_json)
print(user_input)

name='Joe Doe' email='joe.doe@gmail.com' query='I forgot my password' order_id=None purchase_date=None


In [23]:
class CustomerQuery(UserInput):
  priority:str = Field(
    ...,description="Priority level: low,medium,high"
  )
  category:Literal[
    'refund_request','information_request','other'
  ] = Field(
    ...,description="Query category"
  )
  is_complaint:bool = Field(
    ...,description="Whether this is a complaint"
  )
  tags:List[str] = Field(
    ...,description="Relevant keyword tags"
  )

In [27]:
# Create a prompt with generic example data to guide LLM.
example_response_structure = f"""{{
    name="Example User",
    email="user@example.com",
    query="I ordered a new computer monitor and it arrived with the screen cracked. I need to exchange it for a new one.",
    order_id=12345,
    purchase_date="2025-12-31",
    priority="medium",
    category="refund_request",
    is_complaint=True,
    tags=["monitor", "support", "exchange"] 
}}"""

In [28]:
print(example_response_structure)

{
    name="Example User",
    email="user@example.com",
    query="I ordered a new computer monitor and it arrived with the screen cracked. I need to exchange it for a new one.",
    order_id=12345,
    purchase_date="2025-12-31",
    priority="medium",
    category="refund_request",
    is_complaint=True,
    tags=["monitor", "support", "exchange"] 
}


In [29]:
# Create prompt with user data and expected JSON structure
prompt = f"""
Please analyze this user query\n {user_input.model_dump_json(indent=2)}:

Return your analysis as a JSON object matching this exact structure 
and data types:
{example_response_structure}

Respond ONLY with valid JSON. Do not include any explanations or 
other text or formatting before or after the JSON object.
"""

print(prompt)


Please analyze this user query
 {
  "name": "Joe Doe",
  "email": "joe.doe@gmail.com",
  "query": "I forgot my password",
  "order_id": null,
  "purchase_date": null
}:

Return your analysis as a JSON object matching this exact structure 
and data types:
{
    name="Example User",
    email="user@example.com",
    query="I ordered a new computer monitor and it arrived with the screen cracked. I need to exchange it for a new one.",
    order_id=12345,
    purchase_date="2025-12-31",
    priority="medium",
    category="refund_request",
    is_complaint=True,
    tags=["monitor", "support", "exchange"] 
}

Respond ONLY with valid JSON. Do not include any explanations or 
other text or formatting before or after the JSON object.



In [30]:
def call_llm(prompt,model="gemini-3-flash-preview"):
  response = client.models.generate_content(model=model,contents=prompt)
  return response.text

In [32]:
response_content = call_llm(prompt)
print(response_content)

{
    "name": "Joe Doe",
    "email": "joe.doe@gmail.com",
    "query": "I forgot my password",
    "order_id": null,
    "purchase_date": null,
    "priority": "medium",
    "category": "account_access",
    "is_complaint": false,
    "tags": ["password", "account", "login"]
}


In [34]:
valid_data = CustomerQuery.model_validate_json(response_content)

ValidationError: 1 validation error for CustomerQuery
category
  Input should be 'refund_request', 'information_request' or 'other' [type=literal_error, input_value='account_access', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/literal_error